In [1]:
import pandas as pd
from langchain_core.documents import Document
import pickle

In [ ]:
# Load dataset
df = pd.read_csv('Superstore.csv', encoding='cp1252')
# Drop columns not needed based on the test queries
df.drop(columns=['Row ID', 'Order ID', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'Postal Code', 'Product ID'], inplace=True)
df['Order Date'] = pd.to_datetime(df['Order Date'])

In [ ]:
# Global aggregate summaries - business metrics across entire dataset for RAG context
global_aggregate = []

total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
total_orders = len(df)
profit_margin = (total_profit / total_sales * 100) if total_sales > 0 else 0
global_aggregate.append(f"TOTAL AGGREGATE: Sales ${total_sales:,.2f}, Profit ${total_profit:,.2f}, Orders {total_orders:,}, Margin {profit_margin:.2f}%")

In [ ]:
# Yearly totals
yearly_totals = []
df['Year'] = df['Order Date'].dt.year
df['Transaction_Margin'] = (df['Profit'] / df['Sales'] * 100)

yearly_data = df.groupby('Year').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Transaction_Margin': 'mean'  # Average of per-transaction margins
}).reset_index()

for _, row in yearly_data.iterrows():
    year = int(row['Year'])
    sales = row['Sales']
    profit = row['Profit']
    avg_margin = row['Transaction_Margin']
    yearly_totals.append(
        f"Year {year} TOTAL: Sales ${sales:,.2f}, Profit ${profit:,.2f}, Margin {avg_margin:.2f}%"
    )

# Category totals - segment revenue by product category
category_totals = []
for category in sorted(df['Category'].unique()):
    cat_df = df[df['Category'] == category]
    cat_sales = cat_df['Sales'].sum()
    cat_profit = cat_df['Profit'].sum()
    cat_margin = (cat_profit / cat_sales * 100) if cat_sales > 0 else 0
    category_totals.append(
        f"Category {category}: Sales ${cat_sales:,.2f}, Profit ${cat_profit:,.2f}, Margin {cat_margin:.2f}%"
    )

# Region totals - geographic performance breakdown
region_totals = []
for region in ['West', 'East', 'Central', 'South']:
    region_df = df[df['Region'] == region]
    if len(region_df) > 0:
        region_sales = region_df['Sales'].sum()
        region_profit = region_df['Profit'].sum()
        region_margin = (region_profit / region_sales * 100) if region_sales > 0 else 0
        region_totals.append(
            f"Region {region}: Sales ${region_sales:,.2f}, Profit ${region_profit:,.2f}, Margin {region_margin:.2f}%"
        )

# Sub-category totals - granular product segmentation
subcat_totals = []
for sub_cat in sorted(df['Sub-Category'].unique()):
    sub_df = df[df['Sub-Category'] == sub_cat]
    sub_sales = sub_df['Sales'].sum()
    sub_profit = sub_df['Profit'].sum()
    sub_margin = (sub_profit / sub_sales * 100) if sub_sales > 0 else 0
    items_count = len(sub_df)
    discounted_count = len(sub_df[sub_df['Discount'] > 0])
    subcat_totals.append(
        f"Sub-Category {sub_cat}: Sales ${sub_sales:,.2f}, Margin {sub_margin:.2f}%, Items {items_count}, Discounted {discounted_count}"
    )

# Geographic breakdown - top states and cities by sales volume
state_totals = []
state_sales = df.groupby('State').agg({'Sales': 'sum', 'Profit': 'sum'}).reset_index()
state_sales = state_sales.sort_values('Sales', ascending=False).head(10)
for _, row in state_sales.iterrows():
    state = row['State']
    state_s = row['Sales']
    state_p = row['Profit']
    state_m = (state_p / state_s * 100) if state_s > 0 else 0
    state_totals.append(
        f"State {state}: Sales ${state_s:,.2f}, Profit ${state_p:,.2f}, Margin {state_m:.2f}%"
    )

city_totals = []
city_sales = df.groupby('City').agg({'Sales': 'sum', 'Profit': 'sum'}).reset_index()
city_sales = city_sales.sort_values('Sales', ascending=False).head(15)
for _, row in city_sales.iterrows():
    city = row['City']
    city_s = row['Sales']
    city_p = row['Profit']
    city_m = (city_p / city_s * 100) if city_s > 0 else 0
    city_totals.append(
        f"City {city}: Sales ${city_s:,.2f}, Profit ${city_p:,.2f}, Margin {city_m:.2f}%"
    )

# Combine into global summaries for indexing - all dimensions together
global_summaries = global_aggregate + yearly_totals + category_totals + region_totals + subcat_totals + state_totals + city_totals

In [ ]:
# Monthly trends - detect seasonality and track month-over-month patterns
monthly_trends = []

# Top months - identify peak sales periods across all years
df['Month'] = df['Order Date'].dt.month
df['MonthName'] = df['Order Date'].dt.strftime('%B')
top_months = df.groupby(['Month', 'MonthName'])['Sales'].sum().reset_index().sort_values('Sales', ascending=False).head(5)
for _, row in top_months.iterrows():
    monthly_trends.append(f"Peak sales month {row['MonthName']}: ${row['Sales']:,.2f} with high seasonality")
# Year-Month combinations - track performance evolution
df['YearMonth'] = df['Order Date'].dt.to_period('M')
for period in sorted(df['YearMonth'].unique()):
    subset = df[df['YearMonth'] == period]
    sales = subset['Sales'].sum()
    profit = subset['Profit'].sum()
    monthly_trends.append(
        f"{period.strftime('%B %Y')}: Sales ${sales:,.2f}, Profit ${profit:,.2f}"
    )

In [ ]:
# Grouped summaries - time-based and promotional analysis
grouped_summaries = []

grouped_summaries.extend(monthly_trends)

# Discount analysis - which product lines have discount activity
discount_analysis = df[df['Discount'] > 0].groupby('Sub-Category').agg({
    'Discount': 'count',
    'Sales': 'sum'
}).reset_index()
discount_analysis.columns = ['Sub-Category', 'Items_Discounted', 'Total_Sales']
for _, row in discount_analysis.iterrows():
    grouped_summaries.append(
        f"Discount: {row['Sub-Category']} has {row['Items_Discounted']} discounted items, ${row['Total_Sales']:,.2f} in sales"
    )

# Frequently discounted products - identify which items are regularly on sale (20%+ threshold)
df_disc = df[df['Discount'] >= 0.2]
product_disc = df_disc.groupby('Product Name').agg({
    'Discount': ['mean', 'count']
}).reset_index()
product_disc.columns = ['Product Name', 'Avg_Discount', 'Count']
product_disc = product_disc[product_disc['Count'] >= 2]  # At least 2 discounted sales
product_disc = product_disc.sort_values('Avg_Discount', ascending=False)

for _, row in product_disc.iterrows():
    grouped_summaries.append(
        f"Frequently discounted product {row['Product Name']}: Average discount {row['Avg_Discount']*100:.0f}%, {int(row['Count'])} discounted sales"
    )

In [ ]:
# Function to identify if a summary is an aggregate metric (for metadata tagging)
def is_aggregate_metric(text):
    t = text.lower()
    # These are all aggregates that should be indexed
    if any(word in t for word in ["total", "aggregate", "month", "region", "category", "year", "state", "city"]):
        return True
    return False

# Create Document objects with metadata for vector indexing
def semantic_chunk_summaries(summaries, layer_name="global"):
    docs = []
    
    for chunk_id, summary in enumerate(summaries):
        summary = summary.strip()
        if not summary:  
            continue
        
        doc = Document(
            page_content=summary,
            metadata={
                "chunk_id": chunk_id,
                "layer": layer_name,
                "is_aggregate": is_aggregate_metric(summary)
            }
        )
        docs.append(doc)
    
    return docs

In [ ]:
# Convert all summaries to Document objects with metadata for vector embeddings
global_docs = semantic_chunk_summaries(global_summaries, layer_name="global")
grouped_docs = semantic_chunk_summaries(grouped_summaries, layer_name="grouped")

print(f"Global: {len(global_docs)} chunks")
print(f"Grouped: {len(grouped_docs)} chunks")
print(f"Total: {len(global_docs) + len(grouped_docs)} chunks")

Global: 54 chunks
Grouped: 1312 chunks
Total: 1366 chunks


In [ ]:
# Save preprocessed data - two-layer structure for RAG retrieval
# Layer 1: global summaries (totals across all dimensions)
# Layer 2: grouped summaries (temporal and promotional patterns)
data = {
    "global": {"documents": global_docs},
    "grouped": {"documents": grouped_docs}
}

with open("analysis.pkl", "wb") as f:
    pickle.dump(data, f)

In [ ]:
# This section validates that RAG will receive accurate data
df_verify = df.copy()
df_verify['Year'] = df_verify['Order Date'].dt.year
df_verify['Month'] = df_verify['Order Date'].dt.strftime('%B')
df_verify['Profit Margin %'] = (df_verify['Profit'] / df_verify['Sales'] * 100).round(2)

# Q1: Sales trend over 4 years
print("\n[Q1] Sales trend over 4-year period:")
yearly_sales_verify = df_verify.groupby('Year')['Sales'].sum().sort_index()
for year in sorted(yearly_sales_verify.index):
    print(f"  {year}: ${yearly_sales_verify[year]:,.2f}")

# Q2: Monthly seasonality
print("\n[Q2] Highest sales months:")
monthly_sales_verify = df_verify.groupby('Month')['Sales'].sum().sort_values(ascending=False)
for i, (month, sales) in enumerate(monthly_sales_verify.head(5).items(), 1):
    print(f"  {i}. {month}: ${sales:,.2f}")

# Q3: Profit margin over time
print("\n[Q3] Profit margin by year:")
yearly_margin_verify = df_verify.groupby('Year')['Profit Margin %'].mean().sort_index()
for year in sorted(yearly_margin_verify.index):
    print(f"  {year}: {yearly_margin_verify[year]:.2f}%")

# Q4: Category revenue
print("\n[Q4] Category revenue (highest first):")
cat_revenue = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
total_cat_sales = cat_revenue.sum()
for cat, sales in cat_revenue.items():
    pct = (sales / total_cat_sales * 100)
    print(f"  {cat}: ${sales:,.2f} ({pct:.1f}%)")

# Q5: Sub-category margins
print("\n[Q5] Top 5 sub-categories by profit margin:")
sub_cat_verify = df.groupby('Sub-Category').agg({'Sales': 'sum', 'Profit': 'sum'})
sub_cat_verify['Margin %'] = (sub_cat_verify['Profit'] / sub_cat_verify['Sales'] * 100).round(2)
sub_cat_verify = sub_cat_verify.sort_values('Margin %', ascending=False)
for i, (subcat, row) in enumerate(sub_cat_verify.head(5).iterrows(), 1):
    print(f"  {i}. {subcat}: {row['Margin %']:.2f}%")

# Q6: High-discount products
print("\n[Q6] Products with >20% average discount:")
df_disc = df.dropna(subset=['Discount'])
high_disc_prod = df_disc[df_disc['Discount'] >= 0.2].groupby('Product Name').agg({'Discount': 'mean'}).sort_values('Discount', ascending=False)
print(f"  Total products with >20% discount: {len(high_disc_prod)}")
print(f"  Top 3:")
for i, (prod, row) in enumerate(high_disc_prod.head(3).iterrows(), 1):
    print(f"    {i}. {prod}: {row['Discount']*100:.0f}%")

# Q7: Region performance
print("\n[Q7] Region sales performance:")
region_perf = df.groupby('Region').agg({'Sales': 'sum', 'Profit': 'sum'}).sort_values('Sales', ascending=False)
for region, row in region_perf.iterrows():
    print(f"  {region}: ${row['Sales']:,.2f}")

# Q8: State sales
print("\n[Q8] Top 5 states by sales:")
state_sales_verify = df.groupby('State')['Sales'].sum().sort_values(ascending=False)
for i, (state, sales) in enumerate(state_sales_verify.head(5).items(), 1):
    print(f"  {i}. {state}: ${sales:,.2f}")

# Q9: City performance (by profit)
print("\n[Q9] Top 5 cities by profit:")
city_profit_verify = df.groupby('City')['Profit'].sum().sort_values(ascending=False)
for i, (city, profit) in enumerate(city_profit_verify.head(5).items(), 1):
    print(f"  {i}. {city}: ${profit:,.2f}")

# Q10: Category comparison (Technology vs Furniture trends)
print("\n[Q10] Technology vs Furniture sales:")
tech_df = df[df['Category'] == 'Technology']
furn_df = df[df['Category'] == 'Furniture']
tech_total = tech_df['Sales'].sum()
furn_total = furn_df['Sales'].sum()
print(f"  Technology: ${tech_total:,.2f}")
print(f"  Furniture: ${furn_total:,.2f}")

# Q11: West vs East profit
print("\n[Q11] West vs East region comparison:")
west_data = df[df['Region'] == 'West'].agg({'Sales': 'sum', 'Profit': 'sum'})
east_data = df[df['Region'] == 'East'].agg({'Sales': 'sum', 'Profit': 'sum'})
west_margin = (west_data['Profit'] / west_data['Sales'] * 100)
east_margin = (east_data['Profit'] / east_data['Sales'] * 100)
print(f"  West: ${west_data['Profit']:,.2f} profit ({west_margin:.2f}% margin)")
print(f"  East: ${east_data['Profit']:,.2f} profit ({east_margin:.2f}% margin)")


[Q1] Sales trend over 4-year period:
  2014: $484,247.50
  2015: $470,532.51
  2016: $609,205.60
  2017: $733,215.26

[Q2] Highest sales months:
  1. November: $352,461.07
  2. December: $325,293.50
  3. September: $307,649.95
  4. March: $205,005.49
  5. October: $200,322.98

[Q3] Profit margin by year:
  2014: 11.81%
  2015: 11.76%
  2016: 12.98%
  2017: 11.60%

[Q4] Category revenue (highest first):
  Technology: $836,154.03 (36.4%)
  Furniture: $741,999.80 (32.3%)
  Office Supplies: $719,047.03 (31.3%)

[Q5] Top 5 sub-categories by profit margin:
  1. Labels: 44.42%
  2. Paper: 43.39%
  3. Envelopes: 42.27%
  4. Copiers: 37.20%
  5. Fasteners: 31.40%

[Q6] Products with >20% average discount:
  Total products with >20% discount: 1652
  Top 3:
    1. Acco 6 Outlet Guardian Premium Plus Surge Suppressor: 80%
    2. Belkin F9S820V06 8 Outlet Surge: 80%
    3. Acco 6 Outlet Guardian Basic Surge Suppressor: 80%

[Q7] Region sales performance:
  West: $725,457.82
  East: $678,781.24
  C